# Notebook 05 — Optimización de Semáforos con la Fórmula de Webster

**Objetivo:** Calcular el tiempo de ciclo óptimo y la distribución de fases para las
intersecciones semaforizadas de Chapinero usando la fórmula de Webster (1958),
calibrada con los flujos del equilibrio de Wardrop (UE).

## Fórmula de Webster

$$C_0 = \frac{1.5L + 5}{1 - Y}$$

donde:
- $C_0$ = ciclo óptimo (segundos)
- $L$ = tiempo perdido total por ciclo = $\sum_i l_i$ (típico: 3–5 s por fase)
- $Y = \sum_i y_i$ = suma de razones de flujo crítico por fase
- $y_i = q_i / s_i$ = flujo de demanda / flujo de saturación en la fase $i$

**Distribución del tiempo verde:**
$$g_i = (C_0 - L) \cdot \frac{y_i}{Y}$$

**Demora promedio por vehículo** (Webster):
$$d = \frac{C(1-\lambda)^2}{2(1-\lambda x)} + \frac{x^2}{2q(1-x)} - 0.65\left(\frac{C}{q^2}\right)^{1/3} x^{(2+5\lambda)}$$

donde $\lambda = g/C$ (fracción de verde) y $x = q/(s \cdot \lambda)$ (grado de saturación).

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

from src.graph_utils import load_graph

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

print('Librerías cargadas.')

## 1. Cargar datos

In [ ]:
with open(PROCESSED_DIR / 'results_poa.pkl', 'rb') as fh:
    res = pickle.load(fh)

x_ue       = res['x_ue']
t0         = res['t0']
cap        = res['cap']
edges_list = res['edges_list']
poa        = res['poa']

G = load_graph(RAW_DIR / 'chapinero_drive_enriched.graphml')
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

print(f'Grafo cargado: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas')
print(f'PoA del sistema: {poa:.5f}')

## 2. Identificar intersecciones semaforizables

Seleccionamos nodos con:
- Grado de entrada + salida ≥ 3 (intersección real)
- Al menos una arista con flujo UE > 0 (intersección activa en el modelo)

In [ ]:
# Flujos UE por arista en un diccionario
flow_dict = {edges_list[i]: x_ue[i] for i in range(len(edges_list))}

# Nodos con grado total >= 3
candidate_nodes = [
    n for n in G.nodes()
    if G.in_degree(n) + G.out_degree(n) >= 3
]

# Filtrar: al menos una arista de entrada con flujo > 10 veh/h
signal_nodes = []
for n in candidate_nodes:
    incoming_flows = [
        flow_dict.get((u, n, k), 0)
        for u in G.predecessors(n)
        for k in G[u][n]
    ]
    if max(incoming_flows, default=0) > 10:
        signal_nodes.append(n)

print(f'Nodos candidatos (grado ≥ 3)    : {len(candidate_nodes):,}')
print(f'Intersecciones semaforizables   : {len(signal_nodes):,}')

## 3. Modelo de fases por intersección

Asignamos fases según los movimientos de entrada:
- **2 fases** (intersección simple): movimientos N-S y E-O
- **4 fases** (crucero completo): una por cada acceso

En la práctica simplificamos agrupando aristas de entrada en 2 fases opuestas.

In [ ]:
@dataclass
class Phase:
    """Una fase semafórica con sus movimientos de entrada."""
    id: int
    edges: List[Tuple]           # aristas que tienen verde en esta fase
    flow: float = 0.0            # flujo crítico de la fase (veh/h)
    saturation: float = 1800.0   # flujo de saturación (veh/h)
    lost_time: float = 4.0       # tiempo perdido por fase (s)

    @property
    def y(self) -> float:
        """Razón de flujo crítico: y = q / s."""
        return self.flow / self.saturation if self.saturation > 0 else 0


@dataclass
class Intersection:
    """Intersección semaforizada con su plan de señales."""
    node_id: int
    phases: List[Phase]
    cycle: float = 0.0           # ciclo óptimo Webster (s)
    greens: List[float] = field(default_factory=list)  # tiempos verdes (s)
    delay: float = 0.0           # demora promedio Webster (s/veh)
    Y: float = 0.0               # suma de razones críticas
    L: float = 0.0               # tiempo perdido total


def assign_phases(node, G, flow_dict, n_phases=2):
    """Agrupa aristas de entrada en n_phases fases por ángulo de entrada."""
    predecessors = list(G.predecessors(node))
    if not predecessors:
        return []

    node_data = G.nodes[node]
    node_x, node_y = node_data.get('x', 0), node_data.get('y', 0)

    # Calcular ángulo de cada predecesor
    angles = []
    for pred in predecessors:
        pred_data = G.nodes[pred]
        dx = node_x - pred_data.get('x', 0)
        dy = node_y - pred_data.get('y', 0)
        angle = np.degrees(np.arctan2(dy, dx)) % 360
        angles.append(angle)

    # Agrupar en n_phases sectores angulares
    sector_size = 360 / n_phases
    phases = []
    for ph_id in range(n_phases):
        lo = ph_id * sector_size
        hi = lo + sector_size
        phase_edges = []
        phase_flow  = 0
        for i, (pred, angle) in enumerate(zip(predecessors, angles)):
            if lo <= angle < hi:
                for k in G[pred][node]:
                    e = (pred, node, k)
                    f = flow_dict.get(e, 0)
                    phase_edges.append(e)
                    phase_flow = max(phase_flow, f)  # flujo crítico = máximo
        if phase_edges:  # solo añadir fases con movimientos
            phases.append(Phase(id=ph_id, edges=phase_edges, flow=phase_flow))

    # Si no hubo asignación angular útil → 2 fases con todos los accesos
    if len(phases) < 2:
        mid = len(predecessors) // 2
        phases = []
        for ph_id, preds_group in enumerate([predecessors[:mid or 1], predecessors[mid or 1:]]):
            ph_edges, ph_flow = [], 0
            for pred in preds_group:
                for k in G[pred][node]:
                    e = (pred, node, k)
                    f = flow_dict.get(e, 0)
                    ph_edges.append(e)
                    ph_flow = max(ph_flow, f)
            phases.append(Phase(id=ph_id, edges=ph_edges, flow=ph_flow))

    return phases


print('Modelo de fases definido.')

## 4. Fórmula de Webster

In [ ]:
def webster_cycle(phases: List[Phase],
                  C_min: float = 30,
                  C_max: float = 180) -> float:
    """
    Ciclo óptimo de Webster: C0 = (1.5*L + 5) / (1 - Y)
    Clipeado en [C_min, C_max].
    """
    L = sum(ph.lost_time for ph in phases)
    Y = sum(ph.y for ph in phases)
    if Y >= 1.0:
        return C_max   # intersección sobresaturada
    if Y <= 0:
        return C_min
    C0 = (1.5 * L + 5) / (1 - Y)
    return float(np.clip(C0, C_min, C_max))


def webster_greens(phases: List[Phase], cycle: float) -> List[float]:
    """
    Tiempo de verde efectivo por fase:
    g_i = (C - L) * y_i / Y
    """
    L = sum(ph.lost_time for ph in phases)
    Y = sum(ph.y for ph in phases)
    effective_green = cycle - L
    if Y <= 0 or effective_green <= 0:
        return [effective_green / len(phases)] * len(phases)
    return [effective_green * ph.y / Y for ph in phases]


def webster_delay(q: float, s: float, g: float, C: float,
                  l: float = 4.0) -> float:
    """
    Demora promedio por vehículo (Webster 1958, s/veh).
    q = flujo de demanda (veh/s)
    s = flujo de saturación (veh/s)
    g = tiempo verde (s)
    C = ciclo (s)
    """
    q_s   = q / 3600       # veh/s
    s_s   = s / 3600       # veh/s
    lam   = g / C          # fracción de verde
    x     = q_s / (s_s * lam) if (s_s * lam) > 0 else 0.99  # grado de saturación
    x     = min(x, 0.99)   # evitar div/0 cerca de saturación

    # Término 1: demora uniforme
    d1 = C * (1 - lam) ** 2 / (2 * (1 - lam * x))
    # Término 2: demora aleatoria
    d2 = x ** 2 / (2 * q_s * (1 - x)) if (1 - x) > 0 else 0
    # Término 3: corrección empírica de Webster
    d3 = 0.65 * (C / (q_s ** 2 + 1e-9)) ** (1/3) * x ** (2 + 5 * lam)

    return max(d1 + d2 - d3, 0)


print('Funciones de Webster definidas.')

## 5. Aplicar Webster a todas las intersecciones

In [ ]:
intersections: List[Intersection] = []

for node in signal_nodes:
    phases = assign_phases(node, G, flow_dict, n_phases=2)
    if not phases:
        continue

    C0  = webster_cycle(phases)
    gs  = webster_greens(phases, C0)
    L   = sum(ph.lost_time for ph in phases)
    Y   = sum(ph.y for ph in phases)

    # Demora promedio ponderada por flujo
    total_flow  = sum(ph.flow for ph in phases)
    total_delay = 0.0
    for ph, g in zip(phases, gs):
        d = webster_delay(ph.flow, ph.saturation, g, C0, ph.lost_time)
        total_delay += d * ph.flow
    avg_delay = total_delay / total_flow if total_flow > 0 else 0

    inter = Intersection(
        node_id=node,
        phases=phases,
        cycle=C0,
        greens=gs,
        delay=avg_delay,
        Y=Y,
        L=L,
    )
    intersections.append(inter)

print(f'Intersecciones procesadas: {len(intersections)}')

## 6. Estadísticas de los planes de señales

In [ ]:
df_signals = pd.DataFrame([{
    'node':    inter.node_id,
    'n_fases': len(inter.phases),
    'Y':       inter.Y,
    'L':       inter.L,
    'ciclo':   inter.cycle,
    'demora':  inter.delay,
    'flujo_total': sum(ph.flow for ph in inter.phases),
    'lat':     G.nodes[inter.node_id].get('y', np.nan),
    'lon':     G.nodes[inter.node_id].get('x', np.nan),
} for inter in intersections])

print('Estadísticas de los planes de señales:')
display(df_signals[['ciclo','demora','Y','flujo_total']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Ciclo óptimo
ax = axes[0, 0]
ax.hist(df_signals['ciclo'], bins=25, color='#3498db', edgecolor='white', linewidth=0.4)
ax.axvline(df_signals['ciclo'].median(), color='red', linestyle='--',
           label=f'Mediana={df_signals["ciclo"].median():.0f}s')
ax.set_xlabel('Ciclo óptimo Webster (s)', fontsize=11)
ax.set_ylabel('Intersecciones', fontsize=11)
ax.set_title('Distribución de ciclos óptimos')
ax.legend()

# Demora promedio
ax = axes[0, 1]
ax.hist(df_signals['demora'].clip(upper=120), bins=25, color='#e74c3c', edgecolor='white', linewidth=0.4)
ax.axvline(df_signals['demora'].median(), color='navy', linestyle='--',
           label=f'Mediana={df_signals["demora"].median():.1f}s')
ax.set_xlabel('Demora promedio Webster (s/veh)', fontsize=11)
ax.set_ylabel('Intersecciones', fontsize=11)
ax.set_title('Distribución de demoras en intersecciones')
ax.legend()

# Y (grado de saturación agregado)
ax = axes[1, 0]
ax.hist(df_signals['Y'].clip(upper=1), bins=25, color='#2ecc71', edgecolor='white', linewidth=0.4)
ax.axvline(0.85, color='red', linestyle='--', linewidth=1.2, label='Umbral crítico Y=0.85')
ax.set_xlabel('Y (suma de razones de flujo crítico)', fontsize=11)
ax.set_ylabel('Intersecciones', fontsize=11)
ax.set_title('Distribución de Y (saturación agregada)')
ax.legend()

# Ciclo vs demora
ax = axes[1, 1]
sc = ax.scatter(df_signals['ciclo'], df_signals['demora'].clip(upper=120),
                c=df_signals['Y'], cmap='RdYlGn_r', alpha=0.6, s=20)
plt.colorbar(sc, ax=ax, label='Y (saturación)')
ax.set_xlabel('Ciclo óptimo (s)', fontsize=11)
ax.set_ylabel('Demora (s/veh)', fontsize=11)
ax.set_title('Ciclo vs. Demora (color = saturación Y)')

plt.suptitle('Planes de semáforos Webster — Chapinero, Bogotá', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_stats.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Intersecciones críticas

Identificamos las intersecciones con mayor demora y mayor grado de saturación.

In [ ]:
top_delay = df_signals.nlargest(10, 'demora')[['node','ciclo','Y','demora','flujo_total']]
top_Y     = df_signals.nlargest(10, 'Y')[['node','ciclo','Y','demora','flujo_total']]

print('Top 10 intersecciones por DEMORA (s/veh):')
display(top_delay.round(2))

print('\nTop 10 intersecciones por SATURACIÓN (Y):')
display(top_Y.round(4))

## 8. Diagrama de fases para las intersecciones más críticas

In [ ]:
def plot_signal_diagram(inter: Intersection, ax, title=''):
    """Dibuja el diagrama de fases (verde/rojo/amarillo) para una intersección."""
    C   = inter.cycle
    gs  = inter.greens
    yl  = 3   # amarillo (s)
    colors_ph = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

    current = 0
    for ph_id, (ph, g) in enumerate(zip(inter.phases, gs)):
        color = colors_ph[ph_id % len(colors_ph)]
        # Verde
        ax.barh(ph_id, g, left=current, color='#2ecc71', edgecolor='white', linewidth=0.5)
        ax.text(current + g / 2, ph_id, f'{g:.0f}s', ha='center', va='center',
                fontsize=8, color='black', fontweight='bold')
        current += g
        # Amarillo
        ax.barh(ph_id, yl, left=current, color='#f1c40f', edgecolor='white', linewidth=0.5)
        current += yl
        # Tiempo perdido (todo rojo)
        lost = ph.lost_time - yl
        if lost > 0:
            ax.barh(ph_id, lost, left=current, color='#c0392b', edgecolor='white', linewidth=0.5)
            current += lost

    # Rojo para fases que no tienen verde
    ax.set_yticks(range(len(inter.phases)))
    ax.set_yticklabels([f'Fase {ph.id}\ny={ph.y:.3f}' for ph in inter.phases], fontsize=9)
    ax.set_xlabel('Tiempo en ciclo (s)', fontsize=9)
    ax.set_xlim(0, C)
    ax.set_title(f'{title}\nC={C:.0f}s  Y={inter.Y:.3f}  dem={inter.delay:.1f}s/veh',
                 fontsize=9)

    patches = [
        mpatches.Patch(color='#2ecc71', label='Verde'),
        mpatches.Patch(color='#f1c40f', label='Amarillo'),
        mpatches.Patch(color='#c0392b', label='Todo rojo'),
    ]
    ax.legend(handles=patches, fontsize=7, loc='lower right')


# Graficar las 6 intersecciones más críticas por demora
critical = sorted(intersections, key=lambda x: x.delay, reverse=True)[:6]

fig, axes = plt.subplots(3, 2, figsize=(13, 10))
for ax, inter in zip(axes.flatten(), critical):
    plot_signal_diagram(inter, ax, title=f'Nodo {inter.node_id}')

plt.suptitle('Planes de semáforo Webster — Top 6 intersecciones críticas\nChapinero, Bogotá',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_diagrams.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Mapa espacial de ciclos y demoras

In [ ]:
# Asignar atributos al grafo para visualización
for inter in intersections:
    G.nodes[inter.node_id]['webster_cycle'] = inter.cycle
    G.nodes[inter.node_id]['webster_delay'] = inter.delay
    G.nodes[inter.node_id]['webster_Y']     = inter.Y

# Colores y tamaños de nodo
signal_set = {inter.node_id for inter in intersections}
node_sizes  = []
node_colors = []
cmap_delay  = plt.get_cmap('RdYlGn_r')
max_delay   = df_signals['demora'].quantile(0.95)

for n in G.nodes():
    if n in signal_set:
        delay = G.nodes[n].get('webster_delay', 0)
        norm_d = min(delay / max_delay, 1.0)
        node_colors.append(cmap_delay(norm_d))
        node_sizes.append(30 + 60 * norm_d)
    else:
        node_colors.append('#2a2a4a')
        node_sizes.append(3)

fig, ax = ox.plot_graph(
    G,
    node_color=node_colors,
    node_size=node_sizes,
    edge_color='#3a3a5a',
    edge_linewidth=0.6,
    bgcolor='#1a1a2e',
    figsize=(12, 10),
    show=False,
    close=False,
)

sm = plt.cm.ScalarMappable(cmap=cmap_delay,
                            norm=mcolors.Normalize(vmin=0, vmax=max_delay))
sm.set_array([])
plt.colorbar(sm, ax=ax, orientation='vertical', shrink=0.55,
             label='Demora Webster (s/veh)')

ax.set_title('Demora en intersecciones semaforizadas (fórmula de Webster)\nChapinero, Bogotá',
             color='white', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_delay_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Impacto en el costo total del sistema

Estimamos el ahorro al reemplazar tiempos de verde actuales (proporcionales a capacidad)
por los tiempos Webster optimizados.

In [ ]:
# Comparar demora Webster vs. demora con ciclo fijo de referencia (C=90s, verde=50%)
C_ref = 90.0
g_ref_frac = 0.50

rows_impact = []
for inter in intersections:
    for ph, g_opt in zip(inter.phases, inter.greens):
        if ph.flow < 1:
            continue
        g_ref = C_ref * g_ref_frac
        d_opt = webster_delay(ph.flow, ph.saturation, g_opt, inter.cycle, ph.lost_time)
        d_ref = webster_delay(ph.flow, ph.saturation, g_ref, C_ref, ph.lost_time)
        rows_impact.append({
            'node':    inter.node_id,
            'flow':    ph.flow,
            'd_opt':   d_opt,
            'd_ref':   d_ref,
            'saving':  d_ref - d_opt,        # s/veh
            'saving_total': (d_ref - d_opt) * ph.flow,  # veh·s/h
        })

df_impact = pd.DataFrame(rows_impact)
total_saving = df_impact['saving_total'].sum()
avg_saving   = df_impact['saving'].mean()

print('=' * 55)
print('   IMPACTO DE LA OPTIMIZACIÓN WEBSTER')
print('=' * 55)
print(f'  Intersecciones optimizadas  : {len(intersections):,}')
print(f'  Ciclo medio óptimo          : {df_signals["ciclo"].mean():.1f} s')
print(f'  Demora media óptima         : {df_signals["demora"].mean():.1f} s/veh')
print(f'  Demora media ciclo fijo 90s : {df_impact["d_ref"].mean():.1f} s/veh')
print(f'  Ahorro promedio             : {avg_saving:.1f} s/veh')
print(f'  Ahorro total del sistema    : {total_saving:,.0f} veh·s/h')
print(f'                              = {total_saving/3600:,.1f} veh·h/h')
print('=' * 55)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_impact['d_ref'], df_impact['d_opt'],
           c=df_impact['flow'], cmap='plasma', alpha=0.5, s=15)
lim = max(df_impact[['d_ref','d_opt']].max())
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='Sin cambio')
ax.set_xlabel('Demora ciclo fijo 90s (s/veh)', fontsize=11)
ax.set_ylabel('Demora Webster óptimo (s/veh)', fontsize=11)
ax.set_title('Demora: ciclo fijo vs. Webster optimizado\n(puntos bajo la diagonal = mejora)')
ax.legend()
sm = plt.cm.ScalarMappable(cmap='plasma')
sm.set_array(df_impact['flow'])
plt.colorbar(sm, ax=ax, label='Flujo (veh/h)')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'webster_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Resumen ejecutivo del proyecto

In [ ]:
cost_ue  = res['cost_ue']
cost_so  = res['cost_so']

print('=' * 65)
print('   RESUMEN EJECUTIVO — OPTIMIZACIÓN DE TRÁFICO CHAPINERO')
print('=' * 65)
print()
print('1. RED VIAL')
print(f'   Nodos: {G.number_of_nodes():,}  |  Aristas: {G.number_of_edges():,}')
print(f'   Pares O-D modelados: {len(res["od_pairs"])}')
print()
print('2. EQUILIBRIO DE WARDROP (UE)')
print(f'   Costo total: {cost_ue:,.0f} veh·s')
print(f'   Tiempo prom: {cost_ue/sum(d for _,_,d in res["od_pairs"]):.1f} s/veh')
print()
print('3. ÓPTIMO SOCIAL (SO)')
print(f'   Costo total: {cost_so:,.0f} veh·s')
print(f'   Ahorro vs UE: {(cost_ue-cost_so)/3600:,.1f} veh·h/h  ({(poa-1)*100:.2f}%)')
print()
print('4. PRICE OF ANARCHY')
print(f'   PoA = {poa:.5f}  (cota Roughgarden: {res["cota_teorica"]:.4f})')
print(f'   El sistema es {(poa-1)*100:.2f}% menos eficiente que el óptimo central')
print()
print('5. SEMÁFOROS — FÓRMULA DE WEBSTER')
print(f'   Intersecciones optimizadas: {len(intersections):,}')
print(f'   Ciclo medio óptimo        : {df_signals["ciclo"].mean():.1f} s')
print(f'   Demora media optimizada   : {df_signals["demora"].mean():.1f} s/veh')
print(f'   Ahorro vs ciclo fijo 90s  : {avg_saving:.1f} s/veh  ({total_saving/3600:,.1f} veh·h/h)')
print('=' * 65)

## 12. Guardar resultados finales

In [ ]:
results_final = {
    **res,
    'intersections': intersections,
    'df_signals':    df_signals,
    'df_impact':     df_impact,
    'total_saving':  total_saving,
    'avg_saving':    avg_saving,
}

out_path = PROCESSED_DIR / 'results_final.pkl'
with open(out_path, 'wb') as fh:
    pickle.dump(results_final, fh)

# Exportar tabla de señales a CSV
df_signals.to_csv(PROCESSED_DIR / 'signal_plans.csv', index=False)

print(f'Resultados finales guardados en: {out_path}')
print(f'Tabla de planes exportada a: {PROCESSED_DIR}/signal_plans.csv')
print('\nProyecto completado.')